In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU available")

CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.4 GB


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_from_disk
from trl import SFTTrainer
import torch
 

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
 
# LoRA config 
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    inference_mode=False,
)
 
# QLoRA config 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,   # float16, NOT bfloat16
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
 
# Model 
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,          # force float16 explicitly
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Ensure all trainable (LoRA) params are float32 for stable optimizer steps
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
 

# Dataset 
dataset = load_from_disk("data/processed/intent_dataset")
 
# Training args 
training_args = TrainingArguments(
    output_dir="model/intent/checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=10,
    save_steps=200,

    fp16=False,                 # Disable AMP entirely to avoid scaler crash
    bf16=False,
    tf32=False,

    max_grad_norm=1.0,          # Normal clipping, safe now without AMP scaler
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",   # Memory-efficient optimizer for 6GB VRAM
    report_to="none",
)
 

# Trainer 
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
)
 
# TRAIN 
trainer.train()
print("Training complete!")
 
# SAVE 
trainer.model.save_pretrained("model/intent/lora_adapter")
tokenizer.save_pretrained("model/intent/lora_adapter")
print("Adapter saved successfully!")

c:\Users\aa561\.conda\envs\routing-agent\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.676384
20,1.013192
30,0.466764
40,0.198258
50,0.138775
60,0.122789
70,0.115730
80,0.108124
90,0.105669
100,0.105641


Training complete!
Adapter saved successfully!


In [3]:
# Save adapter and tokenizer from current session (no retraining needed)
import os

save_path = "model/intent/lora_adapter"
os.makedirs(save_path, exist_ok=True)

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Verify files were saved
files = os.listdir(save_path)
print(f"Files saved in '{save_path}':")
for f in files:
    size = os.path.getsize(os.path.join(save_path, f))
    print(f"  {f}  ({size/1e6:.2f} MB)")

Files saved in 'model/intent/lora_adapter':
  adapter_config.json  (0.00 MB)
  adapter_model.safetensors  (30.00 MB)
  chat_template.jinja  (0.00 MB)
  README.md  (0.01 MB)
  tokenizer.json  (11.42 MB)
  tokenizer_config.json  (0.00 MB)


In [4]:
# Save adapter, tokenizer, AND checkpoint from current session
import os

save_path = "model/intent/lora_adapter"
checkpoint_path = "model/intent/checkpoints/final"

os.makedirs(save_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

# Save LoRA adapter
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Save full trainer checkpoint (optimizer state, scheduler, training state)
trainer.save_model(checkpoint_path)
trainer.save_state()

# Verify adapter
print("=== LoRA Adapter ===")
for f in os.listdir(save_path):
    size = os.path.getsize(os.path.join(save_path, f))
    print(f"  {f}  ({size/1e6:.2f} MB)")

# Verify checkpoint
print("\n=== Checkpoint ===")
for f in os.listdir(checkpoint_path):
    size = os.path.getsize(os.path.join(checkpoint_path, f))
    print(f"  {f}  ({size/1e6:.2f} MB)")

print("\nAll saved successfully!")

=== LoRA Adapter ===
  adapter_config.json  (0.00 MB)
  adapter_model.safetensors  (30.00 MB)
  chat_template.jinja  (0.00 MB)
  README.md  (0.01 MB)
  tokenizer.json  (11.42 MB)
  tokenizer_config.json  (0.00 MB)

=== Checkpoint ===
  adapter_config.json  (0.00 MB)
  adapter_model.safetensors  (30.00 MB)
  chat_template.jinja  (0.00 MB)
  README.md  (0.01 MB)
  tokenizer.json  (11.42 MB)
  tokenizer_config.json  (0.00 MB)
  training_args.bin  (0.01 MB)

All saved successfully!


In [6]:
import shutil
import os

# Correct paths (project root, not notebooks subfolder)
project_root = r"c:\Users\aa561\Documents\Graduation project\Ai\Routing_Agent"

src_adapter    = os.path.join(os.getcwd(), "model/intent/lora_adapter")
src_checkpoint = os.path.join(os.getcwd(), "model/intent/checkpoints/final")

dst_adapter    = os.path.join(project_root, "model/intent/lora_adapter")
dst_checkpoint = os.path.join(project_root, "model/intent/checkpoints/final")

# Move adapter
shutil.copytree(src_adapter, dst_adapter, dirs_exist_ok=True)
print(f"Adapter copied to:\n  {dst_adapter}")

# Move checkpoint
shutil.copytree(src_checkpoint, dst_checkpoint, dirs_exist_ok=True)
print(f"Checkpoint copied to:\n  {dst_checkpoint}")

# Verify
print("\n=== Verified files in project root ===")
for f in os.listdir(dst_adapter):
    print(f"  {f}")

Adapter copied to:
  c:\Users\aa561\Documents\Graduation project\Ai\Routing_Agent\model/intent/lora_adapter
Checkpoint copied to:
  c:\Users\aa561\Documents\Graduation project\Ai\Routing_Agent\model/intent/checkpoints/final

=== Verified files in project root ===
  adapter_config.json
  adapter_model.safetensors
  chat_template.jinja
  README.md
  tokenizer.json
  tokenizer_config.json


In [7]:
from huggingface_hub import HfApi, login
from dotenv import load_dotenv
import os
 
project_root = r"c:\Users\aa561\Documents\Graduation project\Ai\Routing_Agent"
load_dotenv(os.path.join(project_root, ".env"))

HF_WRITE_TOKEN = os.getenv("HF_WRITE_TOKEN")
HF_LORA_REPO   = os.getenv("MODEL_HF_LORA_REPO")  # "Elbana22/transport-agent-intent-lora"
adapter_path   = os.path.join(project_root, "model", "intent", "lora_adapter")
 
login(token=HF_WRITE_TOKEN) 
api = HfApi()

api.create_repo(repo_id=HF_LORA_REPO, private=True, exist_ok=True)

api.upload_folder(
    folder_path=adapter_path,
    repo_id=HF_LORA_REPO,
    repo_type="model",
)

print(f"Done! → https://huggingface.co/{HF_LORA_REPO}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done! → https://huggingface.co/Elbana22/transport-agent-intent-lora
